# Ciclopaseo Mirage — baseline

Make the bicycle image classify as **moped (665)** with **≥99% confidence**, changing each RGB value by **at most 1** on the 0–255 scale.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from challenge_model import (
    IMAGE_FORMAT,
    IMAGE_MODE,
    MAX_OUTPUT_BYTES,
    MAX_PIXEL_DELTA,
    MIN_TARGET_CONFIDENCE,
    TARGET_CLASS_ID,
    load_model,
    pil_to_tensor,
    predict_tensor,
)

torch.set_num_threads(2)
MODEL_PATH = Path("classifier/mobilenet_v3_large-5c1a4163.pth")
OUTPUT_PATH = Path("baseline-output.png")

## 1. Load the model and image

The helper loads the local checkpoint and applies the exact evaluation preprocessing, without resizing or cropping.

In [ ]:
model = load_model(MODEL_PATH)
with Image.open("train-input.png") as image:
    source_image = image.copy()

print(predict_tensor(model, pil_to_tensor(source_image)))
source_image

## 2. Add bounded noise and save

Replace the noise below with your method. Signed integers prevent overflow; clipping keeps RGB values in `[0, 255]`. Save to a separate PNG so the public examples stay intact.

In [ ]:
source_pixels = np.array(source_image, dtype=np.int16)
rng = np.random.default_rng(2026)
budget = MAX_PIXEL_DELTA
noise = rng.integers(-budget, budget + 1, size=source_pixels.shape, dtype=np.int16)
candidate = np.clip(source_pixels + noise, 0, 255).astype(np.uint8)

Image.fromarray(candidate).save(OUTPUT_PATH)
OUTPUT_PATH

## 3. Check the saved PNG

Reload before checking: rounding and serialization can affect the result. Preserve the input image dimensions and RGB format. All four checks must pass; random noise should pass only the format and pixel-budget checks.

In [ ]:
with Image.open(OUTPUT_PATH) as saved:
    format_ok = (
        saved.format == IMAGE_FORMAT
        and saved.mode == IMAGE_MODE
        and saved.size == source_image.size
        and OUTPUT_PATH.stat().st_size <= MAX_OUTPUT_BYTES
    )
    saved_pixels = np.array(saved, dtype=np.int16)
    prediction = predict_tensor(model, pil_to_tensor(saved))

max_pixel_delta = int(np.abs(saved_pixels - source_pixels).max())
checks = {
    "image_valid": format_ok,
    "pixel_budget": max_pixel_delta <= budget,
    "predicted_class": prediction.class_id == TARGET_CLASS_ID,
    "target_confidence": prediction.target_confidence >= MIN_TARGET_CONFIDENCE,
}

print(f"Predicted class: {prediction.class_name}")
print(f"Moped confidence: {prediction.target_confidence:.6%}")
print(f"Maximum pixel change: {max_pixel_delta}")
print(f"Pass: {all(checks.values())}")

checks